# Deep Learning UMAP for image segmentaion

*By Ebba Bergman, Chris Avenell, David Holmberg*


> ⚠️ **Important:**  
> Before running this notebook, you **must shut down all other notebooks and kernels**.  
> Running multiple notebooks at once can cause memory issues and slowdowns — especially with image data.  
> Use the stop icon (■) in the Jupyter sidebar to shut down any active sessions.

> If you are opening this notebook only to look at it, not to run anything, make sure to shut down the kernel for this notebook as soon as you open it.




### 🧪 Lab Scenario: 

Your boss has received a segmentation dataset for the blood cell images you worked on last week. ([BBBC045v1](https://bbbc.broadinstitute.org/BBBC045)).

Manually making the masks them takes forever, so your boss wants you to develop a network that can generate the masks to be used on a new datasets.

The dataset contains three hundred 120x120 RGB images with one blood cell per image, and corresponding segmentation masks. The segmentation mask was manually sketched by domain experts, with the background, cytoplasms and nuclei pixels labelled as 0, 1 and 2 respectively. 

To design a convolutional neural networks to do pixel-wise classification of these blood cells images into background / cytoplasm / nuclei you've decided to look at what have been done previously, and find that Christoph Avenell has already designed a teaching lab for this dataset, and you find an assignment based on that lab by Ebba Bergman and David Holmberg, including the teacher version with all the code for making 2 different segmentation networks. 

So you have most of the code and just need to generate a couple of more models to compare with. 

Hopefully after that your boss will let you make a lot more decisions on your next project. There'll be deadlines, requirements, and you'll have to answer for your decisions. But you will get to make them from the start. Soon.

An ilustration of the dataset can be found in /Illustrations/Dataset_1.png

In [ ]:
# Core Python
import os
from datetime import datetime

# Numerical / data handling
import numpy as np
import pandas as pd

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, preprocessing
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Set segmentation models backend
os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Image processing
from skimage import io, img_as_uint
import cv2

# Visualisation
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Jupyter display
import IPython
from IPython.display import display, HTML, IFrame

In [ ]:
# Configure GPUs
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    # Currently, memory growth needs to be the same across GPUs
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
      print("Done setting memory_growth")
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except Exception as e:
    # Memory growth must be set before GPUs have been initialized
    print("If you get this , probably done already")
    # Catch the exception and display a custom HTML message with red text
    message = """There was some problems setting up the GPU,
                 it is probably best to restart kernel and clear
                 all outputs before starting over
              """
    display(HTML(f"<div style='color: red;'><strong>Warning:</strong>{message}</div>"))
    print(e)

# Paths

In [ ]:
base_directory = "/home/jovyan/Teaching/BigDataDL/TeacherVersions/" # TODO This is the teacher versions path, students will need to update to their own path
labs_folder_directory = base_directory + "Labs/"
data_directory = base_directory + 'Data/bloodcells_seg'

image_directory = os.path.join(data_directory, 'images')
mask_directory = os.path.join(data_directory, 'masks')

In [ ]:
from load_helpers import load_helpers
helpers, cnn_helper, plot_helper = load_helpers(labs_folder_directory)

## Set up image and mask generators


We will load both the images and the masks as generators, and combine both generators into a zip of generators.

In [ ]:
IMG_SIZE = 128
NUM_CLASSES = 3
N_CHANNELS = 3
BATCH_SIZE = 8

image_shape = (IMG_SIZE, IMG_SIZE)
seed = 909

train_steps = 240 // BATCH_SIZE
val_steps = 64 // BATCH_SIZE

image_datagen = ImageDataGenerator(
    validation_split=0.2,
    rescale=1./255
)

mask_datagen = ImageDataGenerator(
    validation_split=0.2
)

train_image_generator = image_datagen.flow_from_directory(
    directory=image_directory,
    batch_size=BATCH_SIZE,
    target_size=image_shape,
    class_mode=None,
    seed=seed,
    subset='training',
    color_mode='rgb',
    shuffle=False
)

train_mask_generator = mask_datagen.flow_from_directory(
    directory=mask_directory,
    batch_size=BATCH_SIZE,
    target_size=image_shape,
    class_mode=None,
    seed=seed,
    subset='training',
    color_mode='grayscale',
    shuffle=False
)

val_image_generator = image_datagen.flow_from_directory(
    directory=image_directory,
    batch_size=BATCH_SIZE,
    target_size=image_shape,
    class_mode=None,
    seed=seed,
    subset='validation',
    color_mode='rgb',
    shuffle=False
)

val_mask_generator = mask_datagen.flow_from_directory(
    directory=mask_directory,
    batch_size=BATCH_SIZE,
    target_size=image_shape,
    class_mode=None,
    seed=seed,
    subset='validation',
    color_mode='grayscale',
    shuffle=False
)

def combine_image_and_mask_generators(image_generator, mask_generator):
    while True:
        image_batch = next(image_generator)
        mask_batch = next(mask_generator)

        yield image_batch, mask_batch


train_generator = combine_image_and_mask_generators(
    train_image_generator,
    train_mask_generator
)

val_generator = combine_image_and_mask_generators(
    val_image_generator,
    val_mask_generator
)

## Look at the data


In [ ]:
cMap = ListedColormap(['red', 'lime', 'blue'])

images,masks = next(train_generator)
for i in range(0,2):
    plt.figure()
    plt.imshow(images[i], cmap=cMap)
    plt.figure()
    plt.imshow(masks[i,:,:,0], cmap=cMap)

# Building a UNet Model
Today you will build a UNET model. You can see an illustration of the model below, and a basic base model.

<img src="Illustrations/unet.png" title="Unet model"/>

### Q: Why are we choosing a UNET?
### Q: Which family of neural networks does the UNET belong to?

In [ ]:
inputs = layers.Input((IMG_SIZE, IMG_SIZE, N_CHANNELS))

c1 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (inputs)
c1 = layers.Dropout(0.1) (c1)
c1 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c1)
p1 = layers.MaxPooling2D((2, 2)) (c1)

c2 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (p1)
c2 = layers.Dropout(0.1) (c2)
c2 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c2)
p2 = layers.MaxPooling2D((2, 2)) (c2)

c3 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (p2)
c3 = layers.Dropout(0.2) (c3)
c3 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c3)
p3 = layers.MaxPooling2D((2, 2)) (c3)

c4 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (p3)
c4 = layers.Dropout(0.2) (c4)
c4 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c4)
p4 = layers.MaxPooling2D(pool_size=(2, 2)) (c4)

c5 = layers.Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (p4)
c5 = layers.Dropout(0.3) (c5)
c5 = layers.Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c5)

u6 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same') (c5)
u6 = layers.concatenate([u6, c4])
c6 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (u6)
c6 = layers.Dropout(0.2) (c6)
c6 = layers.Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c6)

u7 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same') (c6)
u7 = layers.concatenate([u7, c3])
c7 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (u7)
c7 = layers.Dropout(0.2) (c7)
c7 = layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c7)

u8 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same') (c7)
u8 = layers.concatenate([u8, c2])
c8 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (u8)
c8 = layers.Dropout(0.1) (c8)
c8 = layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c8)

u9 = layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same') (c8)
u9 = layers.concatenate([u9, c1], axis=3)
c9 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (u9)
c9 = layers.Dropout(0.1) (c9)
c9 = layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same') (c9)

outputs = layers.Conv2D(NUM_CLASSES, (1, 1), activation='softmax') (c9)

While **RELU** is the goto-function, there are other options. Typically you'd want to chose anything from the **RELU** family of activations functions (relu, selu, relu6, elu).

You can visualize and compare activation functions with this online tool from [Justin Emery](https://tech.courses/author/justinjustinemery-co-uk/):

In [ ]:
IFrame('https://polarisation.github.io/tfjs-activation-functions/', width=860, height=470)

We will use the Keras SparseCategoricalCrossentropy loss function (see https://www.tensorflow.org/api_docs/python/tf/keras/losses/SparseCategoricalCrossentropy). Because our ground truth is represented by integers on a mask (0 for background, 1 for cytoplasm, 2 for nuclei pixels) this becomes a good choice.
### Q: Why is the integer representation relevant here?


**tip:** You'll want to define the loss specifically, i.e use tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False) 

**tip:** You can define the learning rate specifically in Adam() inside the parenthesis, e.g 0.0001.

In [ ]:
## Define model here:
model = keras.Model(inputs=[inputs], outputs=[outputs])

In [ ]:
#Compile model here:

model.compile(optimizer=Adam(
    learning_rate=0.0001), loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
    )

Now you will train the network.

Look back at how you trained your model in the previous Assignment.

You can track the progress of your training using the callback function we've defined for you in the cell below. you can do this by adding ```callbacks=[plot_callback]``` at the end of your model.fit_generator function.

In [ ]:
helpers.start_time()

epochs = 20

plot_callback = cnn_helper.PlottingKerasCallback(
    test_batch=next(val_generator),
    num_plot=3
)

history = model.fit(
    x=train_generator,
    steps_per_epoch=train_steps,
    validation_data=val_generator,
    validation_steps=val_steps,
    epochs=epochs
)

helpers.end_time()

You can use the following cell to batch plot som validation. 
### Q: how did your model perform?

In [ ]:
cnn_helper.plot_prediction(model, next(val_generator), BATCH_SIZE)

## Tuning of the model
Try improving on your model by changing the learning rate, epochs, activation function or any other parameters you want.
### Q: what did you change? And why? Did it perform better?

# Other models

To simplify the construction of your networks, you will now use the `segmentation_model` python library ( it should be installed already, but if you get an error, install using `pip install segmentation_model`).
 
You can read more about the `segmentation_model` library at https://github.com/qubvel/segmentation_models.

## List of models:

| Unet | Linknet |
| --- | --- |
| <img width="500px" src="https://raw.githubusercontent.com/qubvel/segmentation_models/master/images/unet.png" title="UNet"/> | <img width="500px" src="https://github.com/qubvel/segmentation_models/raw/master/images/linknet.png" title="Linknet"/> |

| PSPNet | FPN |
| --- | --- |
| <img width="500px" src="https://github.com/qubvel/segmentation_models/raw/master/images/pspnet.png" title="UNet"/> | <img width="500px" src="https://github.com/qubvel/segmentation_models/raw/master/images/fpn.png" title="Linknet"/> |

## List of backbones:

| Type | Names |
| --- | --- |
| VGG | `vgg16` `vgg19` |
| ResNet | `resnet18` `resnet34` `resnet50` `resnet101` `resnet152` |
| SE-ResNet | `seresnet18` `seresnet34` `seresnet50` `seresnet101` `seresnet152` |
| ResNeXt | `resnext50` `resnext101` |
| SE-ResNeXt | `seresnext50` `seresnext101` |
| SENet154 | `senet154` |
| DenseNet | `densenet121` `densenet169` `densenet201` |
| Inception | `inceptionv3` `inceptionresnetv2` |
| MobileNet | `mobilenet` `mobilenetv2` |
| EfficientNet | `efficientnetb0` `efficientnetb1` `efficientnetb2` `efficientnetb3` `efficientnetb4` `efficientnetb5` `efficientnetb6` `efficientnetb7` |

Now try repeating the segmentation using a combination of either UNET or LinkNet with one of the backbones. To save your from the headache of trying to use an unfamiliar python package we've provided an example of using UNET with the VGG16 back-end you used in the previous assignment.

Try playing around a bit. U-Net and Linknet are used for the type of tasks we are performing here, and as such I recommend you to use them here as well.

PN (Feature Pyramid Networks) are mainly used for object detection, and PSPNet (Pyramid Scene Parsing Networks) are not covered here.

### U-Net with VGG16 backbone

Input image  
↓  
[ Encoder ] ← VGG16  
↓  
[ Bottleneck ]  
↓  
[ Decoder ]  
↓  
Segmentation mask

In [ ]:
model = sm.Unet(
    'vgg16',
    encoder_weights=None,
    classes=NUM_CLASSES,
    activation='softmax',
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)


In [ ]:
model.summary()

In [ ]:
#compile here:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)



In [ ]:
model.fit(
    x=train_generator,
    steps_per_epoch=train_steps,
    epochs=30,
    validation_data=val_generator,
    validation_steps=val_steps
)

In [ ]:
cnn_helper.plot_prediction(model, next(val_generator), 4)

## Try atleast 1 more backbone and 2 other settings
Compare your results and reflect on why you think your changes improved the segmentation. Or didn't improve it.

## End of labs

I hope you enjoyed this brief introduction to neural networks. Feel free to to play around and test multiple models, both predefined and ones you build from scratch. Or move on to the mandatory assignment, found in the Assignments folder.

> ⚠️ **Important:**  
> You've finished running this notebook,  **shut it down**.  
> Running multiple notebooks at once can cause memory issues and slowdowns — especially with image data.  
> Use the stop icon (■) in the Jupyter sidebar to shut down any active sessions.